# SwiftSQL V3 (可选进阶): 在线执行奖励 GRPO（面向高难多表查询）

> 仅在完成 V1 SFT 和 V2 DPO、希望进一步冲击高难多表查询极限时使用。
> 推荐在 Colab Pro **L4 (24GB)** 或 **A100 (40GB)** 上针对精选的 **300–600 条高难可执行多表样本** 小步运行。


In [ ]:
import os, torch, yaml
from pathlib import Path
from google.colab import drive

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive", force_remount=True)
else:
    print("Google Drive already mounted at /content/drive")
DRIVE_DIR = "/content/drive/MyDrive/swift-sql"
REPO_DIR = "/content/swift-sql"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/2422725977/swift-sql.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

!pip install -q --upgrade unsloth trl datasets sqlglot pyyaml
!pip install -q --no-deps -e {REPO_DIR}
import sys
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

with open(f"{REPO_DIR}/configs/train_config.yaml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)


In [ ]:
# 定义可验证执行奖励函数：JSON 格式 + 多方言语法 + SQLite 沙箱执行匹配 + 简洁度奖励
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer

from swiftsql.data_pipeline import load_jsonl
from swiftsql.dialect import transpile_sql, validate_sql
from swiftsql.evaluation import execute_sqlite_sandbox, results_match
from swiftsql.prompts import parse_model_response

train_records = load_jsonl(Path(DRIVE_DIR) / "data" / "train_sft.jsonl")
hard_exec = [
    r for r in train_records
    if r["task_type"] == "generate"
    and r["complexity"] in ("medium", "complex")
    and r["meta"].get("executable_context")
][:500]
print(f"Selected {len(hard_exec)} hard executable queries for GRPO.")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=f"{DRIVE_DIR}/models/v2_dpo_lora",
    max_seq_length=CFG["model"]["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)

rows = []
for r in hard_exec:
    prompt_msgs = [m for m in r["messages"] if m["role"] != "assistant"]
    rows.append({
        "prompt": tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True),
        "dialect": r["meta"]["dialect"],
        "executable_context": r["meta"]["executable_context"],
        "gold_sqlite_sql": r["meta"].get("gold_sqlite_sql") or r["meta"]["gold_sql"],
    })
grpo_ds = Dataset.from_list(rows)

def execution_reward_func(completions, dialect, executable_context, gold_sqlite_sql, **kwargs):
    rewards = []
    for text, d, ctx, gold in zip(completions, dialect, executable_context, gold_sqlite_sql):
        parsed = parse_model_response(text, default_dialect=d)
        r = 0.0
        if parsed["json_valid"]:
            r += 0.3
        val = validate_sql(parsed["sql"], dialect=d, require_read_only=True)
        if val.valid:
            r += 0.5
            pred_sqlite = transpile_sql(parsed["sql"], source_dialect=d, target_dialect="sqlite")
            gold_out = execute_sqlite_sandbox(ctx, gold)
            if pred_sqlite and gold_out.success:
                pred_out = execute_sqlite_sandbox(ctx, pred_sqlite)
                if pred_out.success and results_match(pred_out.rows, gold_out.rows, order_sensitive="ORDER BY" in gold.upper()):
                    r += 2.0
        if len(text) > 450:
            r -= 0.4
        rewards.append(r)
    return rewards

use_bf16 = torch.cuda.is_bf16_supported()
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[execution_reward_func],
    train_dataset=grpo_ds,
    args=GRPOConfig(
        output_dir=f"{DRIVE_DIR}/checkpoints/v3_grpo",
        learning_rate=1e-5,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_generations=4,
        max_prompt_length=1280,
        max_completion_length=320,
        max_steps=150,
        logging_steps=5,
        save_steps=50,
        fp16=not use_bf16,
        bf16=use_bf16,
        optim="adamw_8bit",
        report_to="none",
    ),
)
trainer.train()

V3_DIR = f"{DRIVE_DIR}/models/v3_grpo_lora"
model.save_pretrained(V3_DIR)
tokenizer.save_pretrained(V3_DIR)
print("Saved V3 GRPO adapter to:", V3_DIR)


In [ ]:
# 释放 Colab Pro GPU 配额（先刷新 Google Drive 缓存，防止权重/GGUF 丢数据）
import os, time
from google.colab import drive, runtime

AUTO_UNASSIGN = True  # 如需保留当前会话继续调试，可改为 False

if AUTO_UNASSIGN:
    print("Syncing and flushing Google Drive before releasing GPU runtime...")
    os.sync()
    time.sleep(5)
    drive.flush_and_unmount()
    print("Google Drive flushed. Releasing Colab GPU runtime now to save compute units!")
    runtime.unassign()
else:
    print("AUTO_UNASSIGN=False: 保持当前会话。用完后请手动点击右上角「断开连接并删除运行时」以停止扣费。")
